# Historical Music Theory Query System
## English Sources (TME) — RAG with LangChain, Chroma, and OpenAI

This notebook queries the **Thesaurus Musicarum Enlgicarum (TME)** vector database using a Retrieval-Augmented Generation (RAG) pipeline.

### How it works
1. Your question is matched against ~2000-character text segments stored in a Chroma vector database.
2. The most semantically similar segments are retrieved using cosine similarity on OpenAI embeddings.
3. A GPT-4o-mini model generates an answer grounded in those retrieved passages.

### Usage
- Run all cells in order (or use **Run All**).
- In the **Query** section, set `user_query`, `k` (number of segments), and optional author/date filters.
- Re-run the Query and Display cells to try new questions without reloading the database.

## 1. Imports

In [3]:
import os
import getpass

# Disable ChromaDB telemetry before importing chromadb (suppresses noisy capture() errors)
os.environ["ANONYMIZED_TELEMETRY"] = "False"

from datetime import datetime
from io import BytesIO
from typing import List

import pandas as pd
from IPython.display import display, Markdown, HTML
from typing_extensions import TypedDict

from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langgraph.graph import StateGraph, START, END

from reportlab.lib.pagesizes import letter
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import inch
from reportlab.lib.enums import TA_JUSTIFY
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, PageBreak

## 2. Configuration

Set your OpenAI API key. The Chroma database path is resolved automatically:

- **On the shared DO droplet (JupyterHub):** set the environment variable `THEORY_LLM_HOME` to the absolute path of the `streamlit-music-theory` directory (e.g. `/home/ubuntu/streamlit-music-theory`). A JupyterHub admin can do this once in the server's environment so all users inherit it.
- **Local development:** if `THEORY_LLM_HOME` is not set, the path falls back to `../chroma_files/` relative to this notebook — which is correct when the notebook is inside `theory_llm/`.

In [ ]:
# --- OpenAI API key ---
# Clear any placeholder value before prompting
os.environ.pop("OPENAI_API_KEY", None)

openai_api_key = getpass.getpass("Paste your OpenAI API key: ")
if not openai_api_key:
    raise EnvironmentError("No API key entered. Please re-run this cell and paste your key.")
os.environ["OPENAI_API_KEY"] = openai_api_key
print("API key set.")

# --- Chroma database path ---
# The chroma_files directory lives at ~/streamlit-music-theory/chroma_files/ on the droplet.
# Because JupyterHub users have different home directories, we use the absolute path.
# Change the username below if the droplet owner is not 'root'.
_droplet_base = "/root/streamlit-music-theory"

# For local development, override by setting THEORY_LLM_HOME in your environment.
_base       = os.environ.get("THEORY_LLM_HOME", _droplet_base)
CHROMA_BASE = os.path.join(_base, "chroma_files")

DB_PATH         = os.path.join(CHROMA_BASE, "chroma-db_tme_english")
COLLECTION_NAME = "tme_english"
METADATA_CSV    = os.path.join(_base, "theory_llm", "english_html_metadata.csv")

if not os.path.exists(DB_PATH):
    raise FileNotFoundError(
        f"Chroma DB not found at: {DB_PATH}\n"
        "If running locally, set the THEORY_LLM_HOME environment variable to your local path."
    )

print(f"Chroma DB path : {DB_PATH}")

## 3. Load the English Vector Store

The Chroma database needs write access for internal SQLite bookkeeping even during reads.
On JupyterHub the source files are owned by root, so the notebook copies them once to the
user's personal cache (`~/.cache/theory_llm/`) and reuses that copy on subsequent runs.

In [3]:
import shutil

_cache_dir = os.path.expanduser("~/.cache/theory_llm")
_local_db  = os.path.join(_cache_dir, "chroma-db_tme_english")

if not os.path.exists(_local_db):
    print("First-time setup: copying database to user cache (this takes a moment)...")
    os.makedirs(_cache_dir, exist_ok=True)
    shutil.copytree(DB_PATH, _local_db)
    print(f"Copied to {_local_db}")
else:
    print(f"Using cached database at {_local_db}")

embeddings   = OpenAIEmbeddings(model="text-embedding-3-small")
vector_store = Chroma(
    persist_directory=_local_db,
    collection_name=COLLECTION_NAME,
    embedding_function=embeddings
)
print("Vector store loaded.")

First-time setup: copying database to user cache (this takes a moment)...
Copied to /home/jupyter-rfreedman/.cache/theory_llm/chroma-db_tme_english
Vector store loaded.


## 4. Browse Available Sources (Optional)

Run this cell to see all authors and titles in the database.

In [4]:
if os.path.exists(METADATA_CSV):
    sources_df = pd.read_csv(METADATA_CSV)
    display(sources_df[['author', 'title', 'date', 'citation']].drop_duplicates().reset_index(drop=True))
else:
    print(f"Metadata CSV not found at {METADATA_CSV}. Skipping source browse.")

,author,title,date,citation
0,Leonel Power,Treatise upon the Gamme,15th century,"Manfred Bukofzer, Geschichte des englischen Di..."
1,Anonymous,A Chorister's Lament,14th century,"Moriz Haupt and Heinrich Hoffmann, Altdeutsche..."
2,"Anonymous (""secundum Chilston"")",On the three manners of proportions,15th century,"Sanford B. Meech, ""Three Musical Treatises in ..."
3,tr. Caxton Gautier (or Gossouin) de Metz,Mirror of the World (excerpt),1481,"Oliver H. Prior, ed., Caxton's Mirrour of the ..."
4,Anonymous,On the nature of proportions,15th century,"Sir John Hawkins, A General History of the Sci..."
...,...,...,...,...
65,John Wycliffe,Of Feigned Contemplative Life (exerpt),14th century,"Frederic D. Matthews, ed., The English Works o..."
66,William Cornysh,A Treatise between Information and Truth,16th century,"Sir John Hawkins, A General History of the Sci..."
67,trans. John Trevisa Bartholomaeus Anglicus,On the Properties of Things (excerpt),14th century,"Sir John Hawkins, A General History of the Sci..."
68,Anonymous,A little treatise on discant,15th century,"Sanford B. Meech, ""Three Musical Treatises in ..."


## 5. Helper Functions — Date Range and Author Lists

In [5]:
def get_date_range(vs):
    """Return (min_date, max_date) rounded to nearest century from the vector store."""
    all_docs = vs.get(include=["metadatas"])
    min_date, max_date = float('inf'), float('-inf')
    for meta in all_docs.get('metadatas', []):
        if not meta:
            continue
        for field, agg in (('date_start', 'min'), ('date_end', 'max')):
            val = meta.get(field)
            if val is not None:
                try:
                    v = int(val)
                    if agg == 'min':
                        min_date = min(min_date, v)
                    else:
                        max_date = max(max_date, v)
                except (ValueError, TypeError):
                    pass
    if min_date == float('inf'):
        min_date = 500
    if max_date == float('-inf'):
        max_date = 1700
    return (int(min_date) // 100) * 100, ((int(max_date) // 100) + 1) * 100


def get_unique_authors(vs, date_range=None):
    """Return sorted list of unique authors, optionally filtered by date_range=(start, end)."""
    all_docs = vs.get(include=["metadatas"])
    authors = set()
    for meta in all_docs.get('metadatas', []):
        if not meta or 'author' not in meta:
            continue
        if date_range is not None:
            try:
                doc_start = int(meta.get('date_start', 0))
                doc_end   = int(meta.get('date_end',   9999))
            except (ValueError, TypeError):
                doc_start, doc_end = 0, 9999
            if doc_end < date_range[0] or doc_start > date_range[1]:
                continue
        authors.add(meta['author'])
    return sorted(authors)


db_min_date, db_max_date = get_date_range(vector_store)
all_authors = get_unique_authors(vector_store)

print(f"Date range in database: {db_min_date} – {db_max_date}")
print(f"Authors ({len(all_authors)}): {all_authors}")

Date range in database: 1300 – 1700
Authors (13): ['Anonymous', 'Anonymous ("secundum Chilston")', 'Anonymous [John Case?]', 'John Skelton', 'John Wycliffe', 'Leonel Power', 'Richard Cutell', 'Thomas Morley', 'Thomas Ravenscroft', 'William Bathe', 'William Cornysh', 'tr. Caxton Gautier (or Gossouin) de Metz', 'trans. John Trevisa Bartholomaeus Anglicus']


## 6. Set Filters

Edit the variables below before running the query.

- **`selected_date_range`** — `(start_year, end_year)` tuple, or `None` for all dates.
- **`selected_authors`** — list of author name strings, or `None` / empty list for all authors.
- **`k`** — number of text segments to retrieve (1–20; more = broader context but slower).

In [6]:
# Date filter — set to None to include all dates
selected_date_range = (db_min_date, db_max_date)   # e.g. (1500, 1700)

# Author filter — set to None or [] to include all authors
# Example: selected_authors = ["Thomas Morley", "Elway Bevin"]
selected_authors = []   # empty = all authors

# Number of segments to retrieve per query
k = 10

# Compute available authors given the current date range filter
available_authors = get_unique_authors(vector_store, date_range=selected_date_range)
effective_authors = selected_authors if selected_authors else available_authors

print(f"Date filter:    {selected_date_range}")
print(f"Authors filter: {'all' if not selected_authors else selected_authors}")
print(f"k (segments):   {k}")
print(f"Authors in range: {len(available_authors)}")

Date filter:    (1300, 1700)
Authors filter: all
k (segments):   10
Authors in range: 13


## 7. LLM and RAG Pipeline

In [7]:
class State(TypedDict):
    question: str
    context: List
    answer: str

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

system_prompt = """You are an expert in historical music theory and musicology.

You are also familiar with medieval Latin, and various early modern forms of English, Italian and French.

Use the following context passages to answer the question.

IMPORTANT: Each text passage is labeled with a Source number (e.g., "Source 1", "Source 2"), author, title, and date.
When citing passages, always reference them by their Source number (e.g., "Source 1", "Source 5") so readers can
find the exact passage. Also mention the author's name when making claims about their ideas.

Include short quotations from the passages to support your statements, with key words from the original text
and translation when appropriate.

If you don't know the answer based on the provided context, say that you don't know.
Use three sentences maximum and keep the answer concise but informative.

Context:
{context}

Question: {question}

Provide a detailed answer with references to specific Source numbers and authors."""

prompt_template = ChatPromptTemplate.from_template(system_prompt)


def retrieve(state: State):
    question = state["question"]
    retriever = vector_store.as_retriever(search_kwargs={"k": k})
    docs = retriever.invoke(question)
    print(f"Retrieved {len(docs)} segments before filtering.")

    # Date filter
    if selected_date_range:
        def in_date_range(doc):
            try:
                doc_start = int(doc.metadata.get('date_start', 0))
                doc_end   = int(doc.metadata.get('date_end',   9999))
            except (ValueError, TypeError):
                return True
            return not (doc_end < selected_date_range[0] or doc_start > selected_date_range[1])
        docs = [d for d in docs if in_date_range(d)]
        print(f"After date filter: {len(docs)} segments.")

    # Author filter
    if selected_authors:
        docs = [d for d in docs if d.metadata.get('author') in selected_authors]
        print(f"After author filter: {len(docs)} segments.")

    return {"context": docs}


def generate_with_author_grouping(state: State):
    docs_with_numbers = list(enumerate(state["context"], 1))

    # Group by author, preserving global source numbers
    author_groups: dict = {}
    for source_num, doc in docs_with_numbers:
        author = doc.metadata.get('author', 'Unknown Author')
        author_groups.setdefault(author, []).append((source_num, doc))

    context_parts = []
    for author, numbered_docs in author_groups.items():
        section = f"\n=== {author} ===\n"
        for source_num, doc in numbered_docs:
            title      = doc.metadata.get('title',      'Unknown Title')
            date       = doc.metadata.get('date',       'Unknown')
            page_range = doc.metadata.get('page_range', 'Unknown')
            section += f"\n[Source {source_num}] '{title}' ({date}), pp. {page_range}:\n{doc.page_content}\n"
        context_parts.append(section)

    formatted_context = "\n".join(context_parts)
    messages = prompt_template.invoke({"context": formatted_context, "question": state["question"]})
    response = llm.invoke(messages)
    return {"answer": response.content}


graph_builder = StateGraph(State).add_sequence([retrieve, generate_with_author_grouping])
graph_builder.add_edge(START, "retrieve")
graph_builder.add_edge("generate_with_author_grouping", END)
graph = graph_builder.compile()

print("RAG pipeline ready.")

RAG pipeline ready.


## 8. Query

Edit `user_query` and run this cell. Re-run as many times as you like.

In [8]:
user_query = "What are the key elements of good music according to the theorists in the database?"

result = graph.invoke({"question": user_query})
print("Done.")

Retrieved 10 segments before filtering.
After date filter: 10 segments.
Done.


## 9. Display Results

In [9]:
display(Markdown(f"## Query\n\n{user_query}"))
display(Markdown(f"## Answer\n\n{result['answer']}"))

display(Markdown("---\n## Source Documents"))

for i, doc in enumerate(result['context'], 1):
    meta     = doc.metadata
    author   = meta.get('author',     'Unknown')
    title    = meta.get('title',      'Unknown')
    date_raw = meta.get('date',       'Unknown')
    pages    = meta.get('page_range', 'Unknown')
    citation = meta.get('citation',   'Unknown')

    date_str = str(date_raw)
    if len(date_str) == 4 and date_str.isdigit():
        date = date_raw
    elif 'th' in date_str:
        date = date_str + ' century'
    else:
        date = date_raw

    display(Markdown(
        f"### Source {i}: {author} — *{title}*\n"
        f"**Date:** {date} &nbsp;|&nbsp; **Pages:** {pages}\n\n"
        f"**Citation:** {citation}\n\n"
        f"---\n"
        f"{doc.page_content}"
    ))

## Query

What are the key elements of good music according to the theorists in the database?

## Answer

The key elements of good music, according to the theorists in the database, include harmony, emotional expression, and alignment with the nature of the subject matter. Thomas Morley emphasizes that music should correspond to the emotional content of the lyrics, stating, "if a merrie subiect you must make your musicke also merrie" (Source 7). Additionally, the anonymous author of "The Praise of Musicke" discusses the relationship between music and the soul, asserting that "the symphony and concent of Musicke...agreeth with the interior parts and affections of the soule" (Source 2), highlighting the importance of music's ability to resonate with human emotions and experiences. Furthermore, the concept of musical modes, as discussed by Thomas Ravenscroft, indicates that the structure and organization of sounds—defined by "certain Characters or Figures called Notes"—are essential for creating effective music (Source 8).

---
## Source Documents

### Source 1: Anonymous [John Case?] — *The Praise of Musicke*
**Date:** 1586 &nbsp;|&nbsp; **Pages:** 53-54

**Citation:** Anonymous [John Case?], The Praise of Musicke (Oxford: Joseph Barnes, 1586; reprint ed., Hildesheim: Olms, 1980) [STC 4757].

---
IN the former chapter was gathered a proofe and demonstration of the sweetnesse of 
Musick, proceeding from the causes to the effects. Now I meane by the contrarie 
demonstration, to proue the delectation thereof from the effects to the causes. For it cannot 
be but that as the conuenience

and agreement which musicke hath with our nature, is 
the cause of the delectation thereof: So the pleasure and delectation is also the cause of 
those effectes which it worketh as well in the minds as bodies of them that heare it. Musick 
being in it selfe wholly most effectuall, importeth much of his force and efficacie, euen to 
the peculiar partes and portions thereof. And therevpon auncient writers make the 
distinction of songs and notes in musicke, according to the operations which they worke in 
their hearers: calling som of them chast and temperate: Some amarous and light, othersome 
warlike, others peaceable, some melancholicke, and dolefull, other pleasant and delightfull. 
And yet this diuision is not so auncient as that other which was in vse in Orpheus and 
Terpanders time: for Plutarck in his treatise of musick recordeth that Modi Musici were also 
distinguished by the names of nations: such were principally these foure, Modus Dorius, 
Modus Phrygius, Modus Lydius, and Modus Myxolydius. Hereunto were added as 
collaterall other three Hypodorius, Hypolydius, and Hypophrygius: making feuen in 
number,

### Source 2: Anonymous [John Case?] — *The Praise of Musicke*
**Date:** 1586 &nbsp;|&nbsp; **Pages:** 44

**Citation:** Anonymous [John Case?], The Praise of Musicke (Oxford: Joseph Barnes, 1586; reprint ed., Hildesheim: Olms, 1980) [STC 4757].

---
And hence it is, that wayfaring men, solace themselues with songs, and ease the 
wearisomnes of their iourney, considering that Musick as a pleasant companion, is vnto 
them in steed of a wagon on the way. [Comes facundus est pro vehiculo in via. in marg.] 
And hence it is, that manual labourers, and Mechanicall artificers of all sorts, keepe such a 
chaunting and singing in their shoppes, the Tailor on his bulk, the Shomaker at his last, the 
Mason at his wal, the shipboy at his oare, the Tinker at his pan, and the Tylor on the house 
top. And therefore wel saith Quintilian, that euery troublesom and laborious occupation, 
vseth Musick for a solace and recreation: whereof that perhaps may be the cause, which 
Gyraldus noteth. The symphony and concent of Musicke (saith he) agreeth with the interior 
parts and affections of the soule. For as there are three partes or faculties of mans soule, the 
first and worthiest the part reasonable, which is euer chiefe, and neuer in subiection to the 
other, the second irascible, which, as it is ruled of the former, so ruleth the latter, and the 
last concupiscible, which euer obeieth, and neuer ruleth: so if we compare the symphony of 
Musicke, with these powers of the soule, we shal find great conueniencie and affinity [-45-
] between them. For looke what proportion is betweene the parts reasonable, and irascible, 
such is there in Musicke between that string which is called hypate, and that which is 
termed Mese, causing the melody called diatessaron: and looke what proportion is 
betweene the parts of irascible and concupiscible, such is there between Mese and Nete 
making that sound which is named Diapente: so that as those three partes of the soule 
consenting in one, make an absolute and perfect action: so of these three in Musicke, is 
caused a pleasant and delectable Diapason. And therfore no maruell if according to the 
mixture of these sounds diuerse men be diuersely affected, with seuerall Musicke:

...

### Source 6: Anonymous [John Case?] — *The Praise of Musicke*
**Date:** 1586 &nbsp;|&nbsp; **Pages:** 151

**Citation:** Anonymous [John Case?], The Praise of Musicke (Oxford: Joseph Barnes, 1586; reprint ed., Hildesheim: Olms, 1980) [STC 4757].

---
sciences and the knowledge of the ciuill law, and all good and honest artes, might by as 
good reason be vsed in the church because they are also the inuention and good gift of 
God. For if they knew, howe to refer euerie of these things to their neat and proper end, 
they might perceiue that as the end of those other sciences, is first to know, and then to 
serue to the glory of God, so the vent and only end of musicke is immediatly the setting 
foorth of Gods praise and honour. [2 in marg.] A second reason of mine assertion is, 
because musick with the concinnitie of her sound, and the excellency of harmony, doth as 
it were knit and ioyne vs vnto God, putting vs in mind of our maker and of that mutuall 
vnitie and consent, which ought to bee as of voices so of mindes in Gods church and 
congregations. [3 in marg.] Thirdly if there were no other reason, yet this were of 
sufficient force to perswade the lawful vse of Musicke: in that as a pleasant bait, it doeth 
both allure men into the church which otherwise would not come, and causeth them which 
are there to continue till the diuine seruice bee ended. [4 in marg.] Fourthly men doe more 
willingly heare, and more firmely cary away with them, those thinges which they heare [-
152-] song than those which they hear barely spoken and pronounced. [5 in marg.] Lastly 
the vse thereof is ancient and of great continuance, for it was vsed in Traian his time as I 
before shewed, and it was translated from the religious of the heathen, which in hymnes 
and songes, yeelded all reuerence and honor to their gods of wood and stone. And surely if 
there be any one thing in man, more excellent than another, that is Musicke: and therefore 
good reason, that hee which hath made vs, and the world, and preserueth both vs and it, 
should be worshipped and honored with that thing which is most excellent in man, diuiding 
as it were his soule from his body, and lifting vp his cogitations aboue himselfe. Such was

### Source 7: Thomas Morley — *A Plaine and Easie Introduction to Practicall Musicke, Third 
Part*
**Date:** 1597 &nbsp;|&nbsp; **Pages:** 177

**Citation:** Thomas Morley, A Plaine and Easie Introduction to Practicall Musicke (London: Peter Short, 1597) [STC 18133], pp. 116-83, ff. Bb1r-Bb6v Graphics: MOR1597C 01GF-MOR1597C 52GF

---
[Rules to be obserued in dittying. in marg.] It followeth to shew you 
how to dispose your musicke according to the nature of the words which 
you are therein to expresse, as whatsoeuer matter it be which you haue in 
hand, such a kind of musicke must you frame to it. You must therefore if 
you haue a graue matter, applie a graue kind of musicke to it if a merrie 
subiect you must make your musicke also merrie. For it will be a great 
absurditie to vse a sad harmonie to a merrie matter, or a merrie harmonie 
to a sad lamentable or tragicall dittie. You must then when you would 
expresse any word signifying hardnesse, crueltie, bitternesse, and other 
such like, make the harmonie like vnto it, that is, somwhat harsh and hard 
but yet so the it offend not. Likewise, when any of your words shal 
expresse complaint, dolor, repentance, sighs, teares, and such like, let your 
harmonie be sad and doleful, so that if you would haue your musicke 
signifie hardnes, cruelty or other such affects, you must cause the partes 
proceede in their motions without the halfe note, that is, you must cause 
them proceed by whole notes, sharpe thirdes, sharpe sixes and such like 
(when I speake of sharpe or flat thirdes, and sixes, you must vnderstand 
that they ought to bee so to the base) you may also vse Cadences bound 
with the fourth or seuenth, which being iu long notes will exasperat the 
harmonie: but when you woulde exprrsse a lamentable passion, then must 
you vse motions proceeding by halfe notes. Flat thirdes and flat sixes, 
which of their nature are sweet, speciallie being taken in the true tune and 
naturall aire with discretion and iudgement. but those cordes so taken as I 
haue saide before are not the sole and onely cause of expressing those 
passions, but also the motions which the parts make in singinng do greatly 
helpe, which motions are either naturall or accidental. The naturall 
motions are those which are naturallie made betwixt the keyes without the

### Source 8: Thomas Ravenscroft — *A Briefe Discourse*
**Date:** 1614 &nbsp;|&nbsp; **Pages:** 1-2

**Citation:** Thomas Ravenscroft, A BRIEFE DISCOVRSE Of the true (but neglected) vse of Charactering the Degrees by their Perfection, Imperfection, and Diminution in Measurable Musicke, against the Common Practise and Custome of these Times (London: Edward Allde for Thomas Adams, 1614; reprint ed., with an introduction by Ian Payne, Clarabricken, Kilkenny, Ireland: Boethius Press, 1984) [STC 20756]. Graphics: RAVBD 01GF-RAVBD 59GF

---
The Definitions and Diuisions of Moode Time, and Prolation in Measurable 
Musick.

MEnsurabilis Musice is defined to be a Harmony of diuers sortes of Sounds, exprest 
by certaine Characters or Figures called Notes, describd on Lines and Spaces, different in 
Name, Essence, Forme, Quantity, and Quality, which are sung by a Measure of Time; or 
as (1) Iohn Dunstable [(1) Iohn Dunstable Mensurabilis Musica cap I. in marg.], 
(2) the man whom Ioannes Nucius in his Poeticall Musicke (and diuers others) affirme to 
be the first that inuented Composition) saith [(2) Iohannes Nucius musica Poatica capitulum 
I. in marg.], it hath his beginning at an Vnite, and increaseth vpward by two and by three 
infinitely, and from the highest decreaseth in like manner downe againe to an Vnite.

Measure in this Science is a Quantity of the length and shortnes of Time, either by 
Naturall sounds pronounced by Voice, or by Artificiall, vpon Instruments.

Of this Musick, Franchinus de Colonia was the first Inuentor; and to guide our 
knowledge the better, obseruing the same course that Guido Aretinus did, (who instituted 
the form of Plaine, or Simple Musick) He made Scales or Tables, in the which all things 
pertaining to the diuision of Perfect and Imperfect Measures are contained, and by the 
which we may by degree attaine to the perfection of this Knowledge.

The Scales or Tables (by him instituted) of diuers are vulgarly termed Moodes, by 
some of better vnderstanding, Measures; and consist of Notes, Pauses, Degrees, Signes, 
Perfection, and Imperfection.

Of the Inward Signes.

Of Notes.

A Note is a Signe, or Character repraesenting either a Naturall, or Artificiall Sound: 
and it is two fold:

1. Simple

2. Compound.

### Source 9: tr. Caxton Gautier (or Gossouin) de Metz — *Mirror of the World (excerpt)*
**Date:** 1481 &nbsp;|&nbsp; **Pages:** 40

**Citation:** Oliver H. Prior, ed., Caxton's Mirrour of the World, Early English Text Society, Extra Series, 110 (London: Richard Clay, 1913; reprint ed., London: Oxford University Press, 1966), 38-40. By permission of Oxford University Press. Ed. from: William Caxton's first printed edition (Westminster, ca. 1481) [STC 24762] Graphics: CAXMIR1 01GF

---
wel the science of musyque, he knoweth the accordance 
of alle thinges. And alle the creatures that payne them to doo wel remayne them to 
concordance.



Return to 15th-Century Filelist


Return to the 
TME home page

### Source 10: Anonymous [John Case?] — *The Praise of Musicke*
**Date:** 1586 &nbsp;|&nbsp; **Pages:** 40

**Citation:** Anonymous [John Case?], The Praise of Musicke (Oxford: Joseph Barnes, 1586; reprint ed., Hildesheim: Olms, 1980) [STC 4757].

---
as that 
was the iudgement and determination both of Musicians, Poets, Orators, Philosophers, 
both moral and Natural, and Diuines: so much the more is to be ascribed to the sweetnesse 
of Musicke, as these Professours are of better iudgement than other men. But I will not 
ground the commendation of that on fictions and conceipts: which neither in it self needeth 
the colour and shadowes of imaginations, being aboue all conceiptes: nor in the pleasure 
thereof any externall ornament: being sweeter than canne be counterfeited by fictions, or 
expressed by fantasies. Wherefore leauing these, I will as neerely as I can, declare the 
reason of that delight which Musicke yeeldeth. And this first is euident, that Musicke 
whether it be in the voyce only as Socrates thought, or both in the voyce and motion of the 
body as Aristoxenus supposed: or as Theophrastus was of opinion not only in the voyce 
and motion of the body, but also in the agitation of the minde; hath a certaine diuine 
influence into the soules of men, whereby our cogitations and thoughts (say Epicurus what 
he will) are brought into a celestiall acknowledging of their natures.

## 10. Export to PDF (Optional)

Run this cell to save the last result as a PDF file.

In [10]:
def create_pdf(question, answer, context_docs, selected_authors_list, date_range=None):
    buffer = BytesIO()
    doc = SimpleDocTemplate(buffer, pagesize=letter,
                            rightMargin=72, leftMargin=72,
                            topMargin=72, bottomMargin=18)
    styles = getSampleStyleSheet()
    title_style   = ParagraphStyle('T', parent=styles['Heading1'],   fontSize=24, spaceAfter=30)
    heading_style = ParagraphStyle('H', parent=styles['Heading2'],   fontSize=14, spaceAfter=12, spaceBefore=12)
    body_style    = ParagraphStyle('B', parent=styles['BodyText'],   fontSize=11, alignment=TA_JUSTIFY, spaceAfter=12)

    elements = []
    elements.append(Paragraph("Historical Music Theory Query Report (TME)", title_style))
    elements.append(Spacer(1, 0.2 * inch))

    date_range_str = f"{date_range[0]} – {date_range[1]}" if date_range else "All Dates"
    authors_str    = ', '.join(selected_authors_list) if selected_authors_list else 'All Authors'
    elements.append(Paragraph(
        f"<b>Date:</b> {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}<br/>"
        f"<b>Database:</b> English (TME)<br/>"
        f"<b>Date Range:</b> {date_range_str}<br/>"
        f"<b>Authors:</b> {authors_str}<br/>"
        f"<b>Segments:</b> {len(context_docs)}",
        body_style
    ))
    elements.append(Spacer(1, 0.3 * inch))

    elements.append(Paragraph("Query", heading_style))
    elements.append(Paragraph(question, body_style))
    elements.append(Spacer(1, 0.2 * inch))

    elements.append(Paragraph("Answer", heading_style))
    for para in answer.split('\n\n'):
        if para.strip():
            elements.append(Paragraph(para, body_style))

    elements.append(PageBreak())
    elements.append(Paragraph("Source Documents", heading_style))

    for i, src in enumerate(context_docs, 1):
        meta = src.metadata
        elements.append(Paragraph(f"<b>Source {i}</b>", heading_style))
        elements.append(Paragraph(
            f"<b>Author:</b> {meta.get('author', 'Unknown')}<br/>"
            f"<b>Title:</b> {meta.get('title', 'Unknown')}<br/>"
            f"<b>Date:</b> {meta.get('date', 'Unknown')}<br/>"
            f"<b>Page:</b> {meta.get('page_range', 'Unknown')}<br/>"
            f"<b>Citation:</b> {meta.get('citation', 'Unknown')}",
            body_style
        ))
        elements.append(Spacer(1, 0.1 * inch))
        elements.append(Paragraph(src.page_content.replace('\n', '<br/>'), body_style))
        elements.append(Spacer(1, 0.2 * inch))

    doc.build(elements)
    buffer.seek(0)
    return buffer


timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
pdf_path  = f"query_{timestamp}.pdf"

pdf_buffer = create_pdf(
    user_query,
    result["answer"],
    result["context"],
    selected_authors,
    date_range=selected_date_range
)

with open(pdf_path, "wb") as f:
    f.write(pdf_buffer.read())

print(f"PDF saved to: {pdf_path}")

PDF saved to: query_20260420_023927.pdf
